In [ ]:
# This is a demonstration of how to use the Google Gemini, Groq, and OpenRouter APIs with Python.

import os

import requests
from dotenv import load_dotenv
from google import genai

load_dotenv(
    override=True
)  # replace old values with the fresh ones from .env  # reads GEMINI_API_KEY, GROQ_API_KEY, OPENROUTER_API_KEY from .env

print("Gemini key loaded    :", os.getenv("GEMINI_API_KEY") is not None)
print("Groq key loaded      :", os.getenv("GROQ_API_KEY") is not None)
print("OpenRouter key loaded:", os.getenv("OPENROUTER_API_KEY") is not None)

Gemini key loaded    : True
Groq key loaded      : True
OpenRouter key loaded: True


In [ ]:
# This code lists all the models available in the Google Gemini API, filtering for text models that contain "flash" in their name and excluding voice (tts), live, and image models.

client = genai.Client()
for model in client.models.list():
    name = model.name
    if "flash" not in name:
        continue
    if "tts" in name or "live" in name or "image" in name:
        continue
    print(name)

models/gemini-2.5-flash
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-omni-1.2-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025


In [ ]:
# This code lists all the models available in the Groq API.

response = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {os.getenv('GROQ_API_KEY')}"},
    timeout=30,  # give up after 30 seconds (Topic 3!)
)
for model in response.json()["data"]:
    print(model["id"])

meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-safeguard-20b
canopylabs/orpheus-v1-english
openai/gpt-oss-120b
qwen/qwen3.8-27b
whisper-large-v3-turbo
allam-2-7b
openai/gpt-oss-20b
canopylabs/orpheus-arabic-saudi
whisper-large-v3
meta-llama/llama-prompt-guard-2-86m


In [ ]:
# This code lists all the models available in the OpenRouter API, filtering for free models (those whose ID ends with ":free").

response = requests.get("https://openrouter.ai/api/v1/models", timeout=30)
for model in response.json()["data"]:
    if model["id"].endswith(":free"):
        print(model["id"])

inclusionai/ling-3.2-flash:free
baidu/ernie-5.5-flash-preview:free
thinkingmachines/inkling-2-small:free
liquid/lfm-3-1.7b:free
particle-ai/particle-1:free
recursive/flint-v1.0:free
nvidia/nemotron-3.5-super-vl:free
dots-studio/dots-3-note-preview:free
nvidia/nemotron-3.5-lightning:free
cohere/north-mini-code:free
nvidia/nemotron-3-ultra-550b-a55b:free
nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free
google/gemma-4-26b-a4b-it:free
google/gemma-4-31b-it:free


In [ ]:
# The above code shows how to list available models from Gemini, Groq, and OpenRouter.
# Next, we will demonstrate how to call each model once and print the answer, token usage, and latency.
# If a model fails (e.g., due to a timeout or rate limit), it will just print the error and move on to the next model.
# In practice, you may want to implement retries for transient errors.

import time

from litellm import completion

FALLBACK_MODELS = [
    "gemini/gemini-3.8-flash",
    "groq/openai/gpt-oss-20b",
    "openrouter/google/gemma-4-31b-it:free",
]

messages = [{"role": "user", "content": "In one sentence, what is an AI agent?"}]

for model in FALLBACK_MODELS:
    print("=" * 60)
    print("Model:", model)
    start = time.perf_counter()
    try:
        response = completion(model=model, messages=messages, timeout=30)
    except Exception as error:
        print("  ERROR:", type(error).__name__, "-", str(error)[:200])
        continue  # try the next model
    seconds = time.perf_counter() - start

    print("  Answer :", response.choices[0].message.content)
    print(
        "  Tokens : in",
        response.usage.prompt_tokens,
        "| out",
        response.usage.completion_tokens,
    )
    print(f"  Latency: {seconds:.2f}s")

Model: gemini/gemini-3.8-flash
  ERROR: Timeout - litellm.Timeout: Read timed out. Timeout passed=30.0, time taken=30.066 seconds
Model: groq/openai/gpt-oss-20b
  Answer : An AI agent is an autonomous software entity that perceives its environment, processes information with artificial‑intelligence techniques, and takes actions to achieve specified goals.
  Tokens : in 81 | out 132
  Latency: 0.71s
Model: openrouter/google/gemma-4-31b-it:free

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm.turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers

  ERROR: RateLimitError - litellm.RateLimitError: RateLimitError: OpenrouterException - {"error":{"message":"Provider returned error","code":429,"metadata":{"raw":"google/gemma-4-31b-it:free is temporarily rate-limited upstrea


In [ ]:
# The above code shows how to call each model once and print the answer, token usage, and latency. However,
# if a model fails (e.g., due to a timeout or rate limit), it will just print the error and move on to the next model. In practice, you may want to implement retries for transient errors.

import litellm

from agentic_ai_120.retry import APIError

litellm.suppress_debug_info = True  # hide LiteLLM's help messages


def call_model_once(model, messages, timeout=30):
    """Make ONE call. Turn any LiteLLM HTTP error into our APIError(code)."""
    try:
        return completion(model=model, messages=messages, timeout=timeout)
    except Exception as error:
        code = getattr(error, "status_code", None)  # LiteLLM errors carry an HTTP code
        if code is None:
            raise  # not an API error (maybe a bug in our code): don't hide it
        raise APIError(code, str(error)[:200]) from error


# Try it on all three models and see which code each one gives
for model in FALLBACK_MODELS:
    try:
        response = call_model_once(model, messages)
        print(f"{model:45} OK   -> {response.choices[0].message.content[:60]}...")
    except APIError as error:
        print(f"{model:45} FAIL -> code {error.code}")

gemini/gemini-3.8-flash                       FAIL -> code 408
groq/openai/gpt-oss-20b                       OK   -> An AI agent is an autonomous system that perceives its envir...
openrouter/google/gemma-4-31b-it:free         FAIL -> code 429


In [ ]:
# Now let's use the call_with_retries() helper function to retry each model a few times before moving on to the next one.

from agentic_ai_120.retry import call_with_retries


def chat_with_fallback(messages, models=FALLBACK_MODELS, max_retries=1, timeout=15):
    """Try each model in order (with retries). Return (answer_text, model_used)."""
    last_error = None

    for model in models:
        print(f"Trying {model} ...")

        def make_call():
            return call_model_once(model, messages, timeout=timeout)

        try:
            response = call_with_retries(make_call, max_retries=max_retries)
        except APIError as error:
            print(f"   failed with {error.code}, moving to the next model")
            last_error = error
            continue  # try the next model in the chain

        return response.choices[0].message.content, model  # success!

    raise last_error  # every model failed


start = time.perf_counter()
answer, model_used = chat_with_fallback(messages)
total = time.perf_counter() - start

print("\nAnswer     :", answer)
print("Answered by:", model_used)
print(f"Total time : {total:.1f}s")

Trying gemini/gemini-3.8-flash ...
   failed with 408, moving to the next model
Trying groq/openai/gpt-oss-20b ...

Answer     : An AI agent is a software system that perceives its environment, processes information, and autonomously takes actions to achieve its goals or solve problems.
Answered by: groq/openai/gpt-oss-20b
Total time : 32.6s


In [ ]:
# Simpe version of privious one, without the call_with_retries() helper function. Just to show how it works.

import random

RETRYABLE_CODES = [408, 429, 500, 503, 504]


def chat_simple(messages, models=FALLBACK_MODELS, max_retries=1, timeout=15, base=1.0):
    """Everything in one place: for each model, try a few times, then move on."""
    last_error = None

    for model in models:  # 1. walk the fallback chain
        for attempt in range(max_retries + 1):  # 2. a few tries per model
            print(f"Trying {model} (attempt {attempt + 1}) ...")
            try:
                response = completion(model=model, messages=messages, timeout=timeout)
                return response.choices[0].message.content, model  # success!
            except Exception as error:
                code = getattr(error, "status_code", None)
                if code is None:
                    raise  # a bug in our code: don't hide it
                last_error = error
                print(f"   failed with {code}")

                if code not in RETRYABLE_CODES:
                    break  # 401, 404...: retrying won't help, go to the next model

                if attempt < max_retries:
                    wait = base * (2**attempt) + random.uniform(0, 1)
                    print(f"   waiting {wait:.1f}s")
                    time.sleep(wait)

        print("   moving to the next model")

    raise last_error  # every model failed


start = time.perf_counter()
answer, model_used = chat_simple(messages)
print(f"\nSIMPLE -> {model_used} in {time.perf_counter() - start:.1f}s")
print("Answer:", answer)

Trying gemini/gemini-3.8-flash (attempt 1) ...
   failed with 408
   waiting 1.2s
Trying gemini/gemini-3.8-flash (attempt 2) ...
   failed with 408
   moving to the next model
Trying groq/openai/gpt-oss-20b (attempt 1) ...

SIMPLE -> groq/openai/gpt-oss-20b in 31.9s
Answer: An AI agent is a software entity that perceives its environment through sensors, processes data to form intentions, and acts on that environment to accomplish specified goals.


In [ ]:
# This is a demonstration of how to log LLM calls to SQLite, including the model used, whether it succeeded, how many tokens were used, how long it took, and how much it cost.

import sqlite3
from datetime import UTC, datetime

DB_PATH = "llm_calls.db"


def init_db(db_path=DB_PATH):
    """Create the log table if it doesn't exist yet."""
    connection = sqlite3.connect(db_path)  # opens the file (creates it if missing)
    connection.execute(
        """
        CREATE TABLE IF NOT EXISTS calls (
            id            INTEGER PRIMARY KEY AUTOINCREMENT,
            created_at    TEXT,
            model         TEXT,
            success       INTEGER,
            error_code    INTEGER,
            input_tokens  INTEGER,
            output_tokens INTEGER,
            latency_ms    REAL,
            cost_usd      REAL
        )
        """
    )
    connection.commit()  # save the change to the file
    connection.close()


def log_call(
    model,
    success,
    error_code,
    input_tokens,
    output_tokens,
    latency_ms,
    cost_usd,
    db_path=DB_PATH,
):
    """Add one row: what happened in one LLM call."""
    connection = sqlite3.connect(db_path)
    connection.execute(
        """
        INSERT INTO calls
            (created_at, model, success, error_code, input_tokens, output_tokens, latency_ms, cost_usd)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            datetime.now(UTC).isoformat(),
            model,
            int(success),
            error_code,
            input_tokens,
            output_tokens,
            latency_ms,
            cost_usd,
        ),
    )
    connection.commit()
    connection.close()


init_db()

# Quick test: write one fake row, then read it back
log_call("test/model", True, None, 10, 20, 123.4, 0.0)

connection = sqlite3.connect(DB_PATH)
for row in connection.execute("SELECT * FROM calls"):
    print(row)
connection.close()

(1, '2026-11-14T20:48:06.912563+00:00', 'test/model', 1, None, 10, 20, 123.4, 0.0)
(2, '2026-11-14T20:59:14.578441+00:00', 'gemini/gemini-3.8-flash', 0, 408, None, None, 15073.112300000503, None)
(3, '2026-11-14T20:59:15.115470+00:00', 'groq/openai/gpt-oss-20b', 1, None, 81, 93, 529.0533000079449, None)
(4, '2026-11-14T20:59:15.809368+00:00', 'openrouter/google/gemma-4-31b-it:free', 0, 429, None, None, 687.7777999907266, None)
(5, '2026-11-14T20:59:49.279749+00:00', 'test/model', 1, None, 10, 20, 123.4, 0.0)


In [ ]:
# This is a demonstration of how to use parameterized queries to prevent SQL injection attacks.
# The example shows both an unsafe way of constructing SQL queries by concatenating strings and a safe way using placeholders.

import sqlite3

connection = sqlite3.connect(":memory:")  # a temporary database that lives only in RAM
connection.execute("CREATE TABLE users (name TEXT, password TEXT)")
connection.execute("INSERT INTO users VALUES ('Rahul', 'rahul123')")
connection.execute("INSERT INTO users VALUES ('Priya', 'priya456')")

evil_input = "' OR '1'='1"

# Way 1: gluing text into the SQL
query = f"SELECT * FROM users WHERE name = '{evil_input}'"
print("SQL that ran:", query)
print("UNSAFE result:", connection.execute(query).fetchall())

# Way 2: ? placeholder, data sent separately
safe_rows = connection.execute(
    "SELECT * FROM users WHERE name = ?", (evil_input,)
).fetchall()
print("SAFE result  :", safe_rows)

SQL that ran: SELECT * FROM users WHERE name = '' OR '1'='1'
UNSAFE result: [('Rahul', 'rahul123'), ('Priya', 'priya456')]
SAFE result  : []


In [ ]:
# This is a demonstration of how to log LLM calls to SQLite, including the model used, whether it succeeded, how many tokens were used, how long it took, and how much it cost.


def get_cost_usd(response):
    """Ask LiteLLM what this call cost in USD. Unknown or free models -> None."""
    try:
        return litellm.completion_cost(completion_response=response)
    except Exception:
        return None  # LiteLLM has no price for this model


def call_model_once(model, messages, timeout=30):
    """Make ONE call, log it to SQLite, and turn LiteLLM errors into APIError(code)."""
    start = time.perf_counter()
    try:
        response = completion(model=model, messages=messages, timeout=timeout)
    except Exception as error:
        latency_ms = (time.perf_counter() - start) * 1000
        code = getattr(error, "status_code", None)
        if code is None:
            raise  # not an API error (maybe a bug in our code): don't hide it
        log_call(model, False, code, None, None, latency_ms, None)  # failed call
        raise APIError(code, str(error)[:200]) from error

    latency_ms = (time.perf_counter() - start) * 1000
    log_call(
        model,
        True,
        None,
        response.usage.prompt_tokens,
        response.usage.completion_tokens,
        latency_ms,
        get_cost_usd(response),
    )  # successful call
    return response


# Basic test: one call to each model, each call gets logged
for model in FALLBACK_MODELS:
    try:
        response = call_model_once(model, messages, timeout=15)
        print(f"{model:45} OK   -> {response.choices[0].message.content[:50]}...")
    except APIError as error:
        print(f"{model:45} FAIL -> code {error.code}")

# Look at what got logged
print()
connection = sqlite3.connect(DB_PATH)
for row in connection.execute(
    "SELECT id, model, success, error_code, input_tokens, output_tokens, latency_ms, cost_usd FROM calls"
):
    print(row)
connection.close()

gemini/gemini-3.8-flash                       FAIL -> code 408
groq/openai/gpt-oss-20b                       OK   -> An AI agent is an autonomous system that senses it...
openrouter/google/gemma-4-31b-it:free         FAIL -> code 429

(1, 'test/model', 1, None, 10, 20, 123.4, 0.0)
(2, 'gemini/gemini-3.8-flash', 0, 408, None, None, 15073.112300000503, None)
(3, 'groq/openai/gpt-oss-20b', 1, None, 81, 93, 529.0533000079449, None)
(4, 'openrouter/google/gemma-4-31b-it:free', 0, 429, None, None, 687.7777999907266, None)


In [ ]:
# This is a demonstration of how to summarize the logged LLM calls in SQLite, including the total number of calls, successful calls, average latency for successful calls, total tokens used, and total cost in USD.

summary_query = """
    SELECT
        model,
        COUNT(*)                                           AS total_calls,
        SUM(success)                                       AS ok_calls,
        AVG(CASE WHEN success = 1 THEN latency_ms END)     AS avg_ok_latency_ms,
        SUM(COALESCE(input_tokens, 0) + COALESCE(output_tokens, 0)) AS total_tokens,
        SUM(COALESCE(cost_usd, 0))                         AS total_cost_usd
    FROM calls
    WHERE model != 'test/model'
    GROUP BY model
    ORDER BY ok_calls DESC
"""

connection = sqlite3.connect(DB_PATH)
rows = connection.execute(summary_query).fetchall()
connection.close()

print(
    f"{'model':42} {'calls':>5} {'ok':>4} {'ok %':>6} {'avg ms':>8} {'tokens':>7} {'cost $':>10}"
)
for model, total, ok, avg_ms, tokens, cost in rows:
    ok_percent = ok / total * 100
    if avg_ms is None:
        avg_text = "-"  # this model never succeeded
    else:
        avg_text = f"{avg_ms:.0f}"
    print(
        f"{model:42} {total:>5} {ok:>4} {ok_percent:>5.0f}% {avg_text:>8} {tokens:>7} {cost:>10.6f}"
    )

model                                      calls   ok   ok %   avg ms  tokens     cost $
groq/openai/gpt-oss-20b                        1    1   100%      529     174   0.000000
openrouter/google/gemma-4-31b-it:free          1    0     0%        -       0   0.000000
gemini/gemini-3.8-flash                        1    0     0%        -       0   0.000000


In [38]:
def stream_model_once(model, messages, timeout=30):
    """Stream ONE call: yield text pieces as they arrive, then log the whole call."""
    start = time.perf_counter()
    try:
        stream = completion(
            model=model, messages=messages, timeout=timeout, stream=True
        )
    except Exception as error:
        latency_ms = (time.perf_counter() - start) * 1000
        code = getattr(error, "status_code", None)
        if code is None:
            raise
        log_call(model, False, code, None, None, latency_ms, None)
        raise APIError(code, str(error)[:200]) from error

    chunks = []  # keep every chunk, so we can count tokens at the end
    first_piece_ms = None

    for chunk in stream:
        chunks.append(chunk)
        piece = chunk.choices[0].delta.content  # the new bit of text (the "delta")
        if piece:
            if first_piece_ms is None:
                first_piece_ms = (
                    time.perf_counter() - start
                ) * 1000  # time to first token
            yield piece  # hand this piece to the caller right away

    latency_ms = (time.perf_counter() - start) * 1000

    # Rebuild one normal response from all the chunks, to get token counts and cost
    full_response = litellm.stream_chunk_builder(chunks, messages=messages)
    log_call(
        model,
        True,
        None,
        full_response.usage.prompt_tokens,
        full_response.usage.completion_tokens,
        latency_ms,
        get_cost_usd(full_response),
    )
    print(f"\n\n[TTFT: {first_piece_ms:.0f} ms | total: {latency_ms:.0f} ms]")


poem_messages = [
    {"role": "user", "content": "Write a 6-line poem about learning AI at night."}
]

for piece in stream_model_once("groq/openai/gpt-oss-20b", poem_messages):
    print(piece, end="", flush=True)

When twilight drifts in through the blinds, I turn my screen to night‑time gold.  
The glow of code whispers like a lullaby, telling stories of circuits old.  
I trace the neurons, neurons, like constellations on a black canvas.  
Each line a comet, each error a star that teaches me to chase.  
Through midnight's quiet I learn algorithms that think and dream.  
And when dawn paints the sky, I sleep with silicon secrets in my seam.

[TTFT: 765 ms | total: 1077 ms]


In [40]:
from dotenv import load_dotenv

from agentic_ai_120.llm_client import chat

load_dotenv(override=True)

messages = [{"role": "user", "content": "In one sentence, what is an AI agent?"}]

text, used = chat(messages)
print(used, "->", text)

pieces, used = chat(messages, stream=True)
print("\nStreaming from", used, ":")
for piece in pieces:
    print(piece, end="", flush=True)

groq/openai/gpt-oss-20b -> An AI agent is a software or robotic entity that autonomously perceives its environment, processes information with artificial intelligence techniques, and takes actions to achieve specified goals.

Streaming from groq/openai/gpt-oss-20b :
An AI agent is a software system that perceives its environment, processes information, and autonomously takes actions to achieve specified goals.